In [1]:
# !python -m pip install spacy

In [2]:
import spacy

In [3]:
# !python -m spacy download en_core_web_sm

In [4]:
nlp = spacy.load('en_core_web_sm') # if error then download 

In [5]:
txt = ("Given the recent downturn in stocks especially in tech which is likely to persist as yields keep going up, "
       "I thought it would be prudent to share the risks of investing in ARK ETFs, written up very nicely by "
       "[The Bear Cave](https://thebearcave.substack.com/p/special-edition-will-ark-invest-blow). The risks comes "
       "primarily from ARK's illiquid and very large holdings in small cap companies. ARK is forced to sell its "
       "holdings whenever its liquid ETF gets hit with outflows as is especially the case in market downturns. "
       "This could force very painful liquidations at unfavorable prices and the ensuing crash goes into a "
       "positive feedback loop leading into a death spiral enticing even more outflows and predatory shorts.")

In [6]:
txt = ("Apple reached an all-time high stock price of 143 dollars this January.")

In [7]:
doc = nlp(txt) # to perform NER

In [8]:
from spacy import displacy

In [9]:
displacy.render(doc, style="ent") # ent for NER

In [10]:
spacy.explain('GPE') # geopolitical entity

'Countries, cities, states'

In [11]:
for entity in doc.ents:
    print(f"{entity.label_}: {entity.text}")

ORG: Apple
MONEY: 143 dollars
DATE: this January


In [12]:
# We're almost there. Now, we need to filter out any entities that are not ORG entities, and append those remaining ORGs to an organization list:

In [13]:
# initialize our list
org_list = []

for entity in doc.ents:
    # if label_ is ORG, we append text, otherwise ignore
    if entity.label_ == 'ORG':
        org_list.append(entity.text)

org_list

['Apple']

In [14]:
# we don't need to see 'ARK' three times, so we use set() to remove duplicates, and then convert back to list
org_list = list(set(org_list))

org_list

['Apple']

## NER using Reddit APIs :https://www.reddit.com/prefs/apps/:

In [15]:
import requests
import pandas as pd


class Reddit:
    def __init__(self, client_id, secret_token, username, password):
        # first create authentication object
        auth = requests.auth.HTTPBasicAuth(client_id, secret_token)
        # build login dictionary
        login = {'grant_type': 'password',
                 'username': username,
                 'password': password}
        # setup header info (incl description of API)
        headers = {'User-Agent': 'NLP_Test1/0.0.1'}
        # send request for OAuth token
        res = requests.post(f'https://www.reddit.com/api/v1/access_token',
                            auth=auth, data=login, headers=headers)
        # pull auth bearer token from response
        token = res.json()['access_token']
        # add authorization to headers dictionary
        headers['Authorization'] = f'bearer {token}'
        # add headers dict to internal attributes
        self.headers = headers
        # and api
        self.api = 'https://oauth.reddit.com'

    def get_new(self, subreddit, iters):
        # initialize dataframe to store data
        df = pd.DataFrame()
        # initialize parameters dictionary
        params = {'limit': 100}
        # iterate through several times to make sure we get all the data available
        for i in range(iters):
            # make request
            res = requests.get(f'{self.api}/r/{subreddit}/new',
                               headers=self.headers,
                               params=params)
            # check that we returned something (if not we reached end)
            if len(res.json()['data']['children']) == 0:
                print('No more found')
                return df
            # iterate through each thread recieved
            for thread in res.json()['data']['children']:
                # add info to dataframe
                df = df.append({
                    'id': thread['data']['name'],
                    'created_utc': int(thread['data']['created_utc']),
                    'subreddit': thread['data']['subreddit'],
                    'title': thread['data']['title'],
                    'selftext': thread['data']['selftext'],
                    'upvote_ratio': thread['data']['upvote_ratio'],
                    'ups': thread['data']['ups'],
                    'downs': thread['data']['downs'],
                    'score': thread['data']['score']
                }, ignore_index=True)
            # get earliest ID
            earliest = df['id'].iloc[len(df)-1]
            # add earliest ID to params
            params['after'] = earliest
        return df

In [16]:
SUB = 'investing'

In [17]:
CLIENT_ID = ''
SECRET_TOKEN = ''

In [18]:
USER = 'tech-know-l0G'
PWD = ''

In [19]:
reddit = Reddit(CLIENT_ID, SECRET_TOKEN, USER, PWD)

In [20]:
data = reddit.get_new(SUB, 20)

No more found


In [22]:
datastore_NER = "D:/2022/My/STUDY/NLP_Udemy_JamesBriggs/Resources/datastore/NER/"

In [25]:
data.to_csv(f'{datastore_NER}data/reddit_{SUB}.csv', sep='|', index=False)

In [26]:
import spacy
import pandas as pd

In [27]:
nlp = spacy.load('en_core_web_sm')

In [38]:
BLACKLIST = ['ev', 'covid', 'etfs', 'nyse', 'sec', 'spac', 'fda']
def get_orgs(text):
    # process the text with our SpaCy model to get named entities
    doc = nlp(text)
    # initialize list to store identified organizations
    org_list = []
    # loop through the identified entities and append ORG entities to org_list
    for entity in doc.ents:
        if entity.label_ == 'ORG' and entity.text.lower() not in BLACKLIST:
            org_list.append(entity.text)
    # if organization is identified more than once it will appear multiple times in list
    # we use set() to remove duplicates then convert back to list
    org_list = list(set(org_list))
    return org_list

In [29]:
df = pd.read_csv(datastore_NER+'/data/reddit_investing.csv', sep='|')
df.head()

,created_utc,downs,id,score,selftext,subreddit,title,ups,upvote_ratio
0,1.653959e+09,0.0,t3_v1ejso,1.0,My salary has not gone up at all. I got a 2% r...,investing,If they are trying to bring down inflation to ...,1.0,0.67
1,1.653956e+09,0.0,t3_v1dkf6,2.0,So I've been super hyped about leveraged ETFs ...,investing,Is there something I'm missing? Leverage ETFs ...,2.0,0.63
2,1.653954e+09,0.0,t3_v1d06q,5.0,Hi everyone. Got a freeriding violation on my ...,investing,Is there any monetary penalty to a freeriding ...,5.0,0.86
3,1.653952e+09,0.0,t3_v1c97e,2.0,Dimensional Fund Advisors is an investment adv...,investing,Thoughts on Dimensional Fund Advisors’ investm...,2.0,1.00
4,1.653951e+09,0.0,t3_v1bz2b,42.0,Currently looking to start DCAing into a numbe...,investing,Investing In ETFs in the current market. What'...,42.0,0.89


In [39]:
df['organizations'] = df['selftext'].apply(get_orgs)
df.tail()

,created_utc,downs,id,score,selftext,subreddit,title,ups,upvote_ratio,organizations
918,1.649178e+09,0.0,t3_twzgkr,0.0,Current price in USD **$2**HIVE is a cryptocur...,investing,"Take a look at HIVE, seems undervalued, doesn'...",0.0,0.47,"[CAD, RSI(43, NASDAQ]"
919,1.649176e+09,0.0,t3_twyyg5,51.0,To me the biggest effect of the pandemic is ju...,investing,Labor shortage (Boomers retiring) and how it w...,51.0,0.77,[’ve]
920,1.649173e+09,0.0,t3_twxm95,5.0,I understand how with dividend stock you can r...,investing,How can compound interest be used with index f...,5.0,0.58,[VOO]
921,1.649168e+09,0.0,t3_twvtd0,5.0,"hi, \n\nI've recently been picking up the basi...",investing,How do I do financial projections for DCF?,5.0,0.78,[]
922,1.649165e+09,0.0,t3_twuqsn,0.0,"Hi guys,\n\nAnyone ever come across a whitepap...",investing,Business cycle leading indicators?,0.0,0.33,[]


In [40]:
# merge organizations column into one big list
orgs = df['organizations'].to_list()
orgs = [org for sublist in orgs for org in sublist]
orgs[:10]

['DFSV',
 'Dimensional Fund Advisors',
 'DFA',
 'AUM',
 'CIO of Dimensional Fund Advisors',
 'DBA',
 'VDE',
 'VOO',
 'SWPPX',
 'FXAIX']

In [41]:
from collections import Counter
# create dictionary of organization mention frequency
org_freq = Counter(orgs)
org_freq.most_common(10)

[('Fed', 45),
 ('Amazon', 23),
 ('Vanguard', 22),
 ('VOO', 19),
 ('Fidelity', 15),
 ('SPY', 15),
 ('Apple', 15),
 ('Tesla', 13),
 ('Microsoft', 11),
 ('Treasury', 10)]

In [43]:
df.to_csv(datastore_NER+'/data/processed/reddit_investing_ner.csv', sep='|', index=False)

In this section we will work through applying basic sentiment analysis to our data using a pre-built distilBERT model from the Flair library. We will then use our organization labels captured through NER in the previous section to create a list of organizations with the highest and lowest average sentiment scores.

In [62]:
import pandas as pd
import flair

In [63]:
model = flair.models.TextClassifier.load('en-sentiment')

2022-05-31 13:37:22,513 loading file C:\Users\upadh\.flair\models\sentiment-en-mix-distillbert_4.pt


For each sample there are a few steps we need to take to create the sentiment score. We need to tokenize the input text, make a prediction, extract the direction (positive or negative) and confidence (a score from 0 to 1). If this is new to you, we cover the Flair sentiment model in more depth in TK insert link.

The following function carries out each of these steps for a single extract:

In [64]:
def get_sentiment(text):
    # tokenize input text
    sentence = flair.data.Sentence(text)
    # make sentiment prediction
    model.predict(sentence)
    # extract sentiment direction and confidence (label and score) object
    sentiment = sentence.labels[0]
    return sentiment

In [65]:
# load data
df = pd.read_csv(datastore_NER+'/data/processed/reddit_investing_ner.csv', sep='|')
df.head()

,created_utc,downs,id,score,selftext,subreddit,title,ups,upvote_ratio,organizations
0,1.653959e+09,0.0,t3_v1ejso,1.0,My salary has not gone up at all. I got a 2% r...,investing,If they are trying to bring down inflation to ...,1.0,0.67,[]
1,1.653956e+09,0.0,t3_v1dkf6,2.0,So I've been super hyped about leveraged ETFs ...,investing,Is there something I'm missing? Leverage ETFs ...,2.0,0.63,[]
2,1.653954e+09,0.0,t3_v1d06q,5.0,Hi everyone. Got a freeriding violation on my ...,investing,Is there any monetary penalty to a freeriding ...,5.0,0.86,[]
3,1.653952e+09,0.0,t3_v1c97e,2.0,Dimensional Fund Advisors is an investment adv...,investing,Thoughts on Dimensional Fund Advisors’ investm...,2.0,1.00,"['DFSV', 'Dimensional Fund Advisors', 'DFA', '..."
4,1.653951e+09,0.0,t3_v1bz2b,42.0,Currently looking to start DCAing into a numbe...,investing,Investing In ETFs in the current market. What'...,42.0,0.89,"['DBA', 'VDE']"


In [66]:
# get sentiment
df['sentiment'] = df['selftext'].apply(get_sentiment)
df.head()

,created_utc,downs,id,score,selftext,subreddit,title,ups,upvote_ratio,organizations,sentiment
0,1.653959e+09,0.0,t3_v1ejso,1.0,My salary has not gone up at all. I got a 2% r...,investing,If they are trying to bring down inflation to ...,1.0,0.67,[],NEGATIVE (1.0)
1,1.653956e+09,0.0,t3_v1dkf6,2.0,So I've been super hyped about leveraged ETFs ...,investing,Is there something I'm missing? Leverage ETFs ...,2.0,0.63,[],NEGATIVE (0.9968)
2,1.653954e+09,0.0,t3_v1d06q,5.0,Hi everyone. Got a freeriding violation on my ...,investing,Is there any monetary penalty to a freeriding ...,5.0,0.86,[],POSITIVE (0.9948)
3,1.653952e+09,0.0,t3_v1c97e,2.0,Dimensional Fund Advisors is an investment adv...,investing,Thoughts on Dimensional Fund Advisors’ investm...,2.0,1.00,"['DFSV', 'Dimensional Fund Advisors', 'DFA', '...",POSITIVE (0.9315)
4,1.653951e+09,0.0,t3_v1bz2b,42.0,Currently looking to start DCAing into a numbe...,investing,Investing In ETFs in the current market. What'...,42.0,0.89,"['DBA', 'VDE']",POSITIVE (0.9662)


Now we need to extract each of the organizations alongside it's sentiment score. We will then loop through each, tallying up a total sentiment score and count.

Before we do that, we need to convert each value in the organizations column to a list (they are currently strings because we cannot save Python lists to file within Pandas dataframes, they are automatically converted to strings).

In [67]:
import ast

df['organizations'] = df['organizations'].apply(lambda x: ast.literal_eval(x))

In [104]:
# initialize sentiment dictionary
sentiment = {}

# loop through dataframe and extract org labels and sentiment scores into sentiment dictionary
for i, row in df.iterrows():
    # extract sentiment direction and score
    direction = row['sentiment'].value
    score = row['sentiment'].score
    # loop through each label in organizations column
    for org in row['organizations']:
        # check if org label exists in sentiment dictionary already
        if org not in sentiment.keys():
            # if it doesn't, initialize new entry in dictionary
            sentiment[org] = {'POSITIVE': [], 'NEGATIVE': []}
        # append positive/negative score to respective dictionary entry
        sentiment[org][direction].append(score)

In [105]:
# Now we can loop through each organization entry in the sentiment dictionary and calculate an average positive, and average negative score:
# initialize sentiment list
avg_sentiment = []

# loop through each organization
for org in sentiment.keys():
    # get number of positive and negative ratings
    pos_freq = len(sentiment[org]['POSITIVE'])
    neg_freq = len(sentiment[org]['NEGATIVE'])
    freq =  pos_freq + neg_freq
    for direction in ['POSITIVE', 'NEGATIVE']:
        # assign to variable for cleaner code
        score = sentiment[org][direction]
        # if there are no entries, set to 0
        if len(score) == 0:
            sentiment[org][direction] = 0.0
        else:
            # otherwise calculate total
            sentiment[org][direction] = sum(score)
    # now calculate total amount
    total = sentiment[org]['POSITIVE'] - sentiment[org]['NEGATIVE']
    # and the average score
    avg = total/freq
    pos_score = (sentiment[org]['POSITIVE'] / pos_freq) if pos_freq != 0 else 0
    neg_score = (sentiment[org]['NEGATIVE'] / neg_freq) if neg_freq != 0 else 0
    # add to sentiment list
    avg_sentiment.append({
        'entity': org,
        'positive': pos_score,
        'negative': neg_score,
        'frequency': freq,
        'score': avg
    })

In [106]:
sentiment_df = pd.DataFrame(avg_sentiment, avg_sentiment)
sentiment_df.tail()

,entity,positive,negative,frequency,score
1058,FSI,0.0,0.987442,1,-0.987442
1059,The Dow Transports,0.0,0.987442,1,-0.987442
1060,HDFC,0.0,0.999988,1,-0.999988
1061,CAD,0.0,0.521009,1,-0.521009
1062,RSI(43,0.0,0.521009,1,-0.521009


In [107]:
sentiment_df = sentiment_df[sentiment_df['frequency'] > 3]
sentiment_df

,entity,positive,negative,frequency,score
7,VOO,0.906344,0.893251,19,-0.514389
12,Fed,0.753533,0.975098,45,-0.859856
15,Atlanta Fed,0.837226,0.997813,4,-0.539053
24,Reuters,0.000000,0.997054,6,-0.997054
30,Amazon,0.857380,0.963665,23,-0.567786
34,the Federal Reserve,0.837226,0.978183,9,-0.776471
37,Tesla,0.973939,0.994012,13,-0.842631
42,Fidelity,0.802767,0.966255,15,-0.730386
43,Vanguard,0.843154,0.963299,22,-0.881187
50,HSA,0.000000,0.973655,6,-0.973655


In [109]:
sentiment_df.sort_values('score').head(20)

,entity,positive,negative,frequency,score
201,NAV,0.0,0.999980,4,-0.999980
481,ARKK,0.0,0.998511,4,-0.998511
154,Morgan Stanley,0.0,0.998241,4,-0.998241
101,DRIP,0.0,0.997777,4,-0.997777
24,Reuters,0.0,0.997054,6,-0.997054
330,EU,0.0,0.995443,5,-0.995443
68,AAPL,0.0,0.992746,7,-0.992746
155,Meta,0.0,0.990400,9,-0.990400
384,Goldman Sachs,0.0,0.989011,8,-0.989011
76,ATH,0.0,0.986930,4,-0.986930


## NER and Transformers using spacy

In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2021 NVIDIA Corporation
Built on Sun_Mar_21_19:24:09_Pacific_Daylight_Time_2021
Cuda compilation tools, release 11.3, V11.3.58
Build cuda_11.3.r11.3/compiler.29745058_0


In [5]:
# !python -m pip install -U setuptools pip

In [6]:
# !python -m pip install cupy-cuda113

In [7]:
!python -m pip show cupy-cuda113

Name: cupy-cuda113
Version: 10.5.0
Summary: CuPy: NumPy & SciPy for GPU
Home-page: https://cupy.dev/
Author: Seiya Tokui
Author-email: tokui@preferred.jp
License: MIT License
Location: c:\users\upadh\anaconda3\envs\nlp\lib\site-packages
Requires: fastrlock, numpy
Required-by: 


In [9]:
!python -m pip install -U spacy[cupy-cuda113]
# https://github.com/explosion/spaCy/issues/3496
# https://github.com/explosion/spaCy/issues/5368

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy-transformers 1.0.2 requires spacy<3.1.0,>=3.0.0, but you have spacy 3.3.0 which is incompatible.
en-core-web-trf 3.0.0 requires spacy<3.1.0,>=3.0.0, but you have spacy 3.3.0 which is incompatible.


  Using cached spacy-3.3.0-cp38-cp38-win_amd64.whl (12.0 MB)
  Using cached thinc-8.0.16-cp38-cp38-win_amd64.whl (1.1 MB)
  Attempting uninstall: thinc
    Found existing installation: thinc 8.0.3
    Uninstalling thinc-8.0.3:
      Successfully uninstalled thinc-8.0.3
  Attempting uninstall: spacy
    Found existing installation: spacy 3.0.6
    Uninstalling spacy-3.0.6:
      Successfully uninstalled spacy-3.0.6



en-core-web-sm 3.0.0 requires spacy<3.1.0,>=3.0.0, but you have spacy 3.3.0 which is incompatible.


In [10]:
import spacy
spacy.require_gpu()

C:\Users\upadh\anaconda3\envs\NLP\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\Users\upadh\anaconda3\envs\NLP\lib\site-packages\numpy\.libs\libopenblas.FB5AE2TYXYH2IJRDKGDGQ3XBKLKTF43H.gfortran-win_amd64.dll
C:\Users\upadh\anaconda3\envs\NLP\lib\site-packages\numpy\.libs\libopenblas.WCDJNK7YVMPZQ2ME2ZZHJJRJ3JIKNDB7.gfortran-win_amd64.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


True

In [11]:
!python -m spacy download en_core_web_trf

     -------------------------------------- 460.3/460.3 MB 1.2 MB/s eta 0:00:00
     -------------------------------------- 51.4/51.4 kB 663.2 kB/s eta 0:00:00
  Attempting uninstall: spacy-transformers
    Found existing installation: spacy-transformers 1.0.2
    Uninstalling spacy-transformers-1.0.2:
      Successfully uninstalled spacy-transformers-1.0.2
  Attempting uninstall: en-core-web-trf
    Found existing installation: en-core-web-trf 3.0.0
    Uninstalling en-core-web-trf-3.0.0:
      Successfully uninstalled en-core-web-trf-3.0.0
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


2022-06-01 11:25:10.182210: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library cudart64_110.dll
C:\Users\upadh\anaconda3\envs\NLP\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\Users\upadh\anaconda3\envs\NLP\lib\site-packages\numpy\.libs\libopenblas.FB5AE2TYXYH2IJRDKGDGQ3XBKLKTF43H.gfortran-win_amd64.dll
C:\Users\upadh\anaconda3\envs\NLP\lib\site-packages\numpy\.libs\libopenblas.WCDJNK7YVMPZQ2ME2ZZHJJRJ3JIKNDB7.gfortran-win_amd64.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [12]:
from spacy import displacy

In [13]:
trf = spacy.load('en_core_web_trf')

C:\Users\upadh\anaconda3\envs\NLP\lib\site-packages\spacy\util.py:1639: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)


In [14]:
!python -m pip show cupy

In [15]:
doc = trf("Apple reached an all-time high stock price of 143 dollars this January.")

In [16]:
displacy.render(doc, style='ent')

In [17]:
txt = "Total nonfarm payroll employment rose by 266,000 in April, and the unemployment rate was little changed at 6.1 percent, the U.S. Bureau of Labor Statistics reported today. Notable job gains in leisure and hospitality, other services, and local government education were partially offset by employment declines in temporary help services and in couriers and messengers."

In [18]:
doc = trf(txt)  # en_core_web_trf
displacy.render(doc, style='ent')

In [19]:
txt = """Fastly released its Q1-21 performance on Thursday, after which the stock price dropped a whopping 27%. The company generated revenues of $84.9 million (35% YoY) vs. $85.1 million market consensus. Net loss per share was $0.12 vs. an expected $0.11.

These are not big misses but make the company one of the few high-growth cloud players that underperformed market expectations.

However, the company also lowered its guidance for Q2: Fastly forecasts revenues of $84 - $87 million and a net loss of $0.16 - $0.19 per share, compared to the market consensus of $92 million in revenue and a net loss of $0.08 per share, thereby disappointing investors.

Lastly, Adriel Lares will step down as CFO of the company after 5 years."""

In [20]:
doc = trf(txt)  # en_core_web_trf
displacy.render(doc, style='ent')